# Studi Kasus HCI & Visualisasi Data
## Perancangan Dashboard Pemantauan Harga Pangan **"SIPANGAN Bengkulu"** Berbasis *User-Centered Design*

**Tugas Kuliah Praktisi Informatika**
Materi: *Human-Computer Interaction (HCI) dan Data Visualization*
Narasumber: Dr. Arafat Febriandirza, S.T., M.T.I. (BRIN)
Program Studi S1 Teknik Informatika, Fakultas Teknik, Universitas Bengkulu

| | |
|---|---|
| **Nama** | ........................................ |
| **NPM** | ........................................ |
| **Kelas** | ........................................ |

---

### Daftar Isi
1. Pendahuluan (latar belakang, rumusan masalah, tujuan, metodologi)
2. Tahap 1 — Analisis Kebutuhan Pengguna (*User-Centered Design*)
3. Tahap 2 — Konsep Desain Berbasis Psikologi Kognitif (*Recognition over Recall*, Gestalt)
4. Tahap 3 — Penyiapan Data (pembangkitan dan pembersihan data)
5. Tahap 4 — Prototipe *Low-Fidelity* (wireframe)
6. Tahap 5 — Prinsip Visualisasi Data (Tufte, pemilihan jenis grafik, aksesibilitas)
7. Tahap 6 — Prototipe *High-Fidelity*: Dashboard Interaktif (mantra Shneiderman)
8. Tahap 7 — Evaluasi *Usability* (uji tugas dan *System Usability Scale*)
9. Tahap 8 — Iterasi Desain Berdasarkan Hasil Evaluasi
10. Kesimpulan
11. Daftar Pustaka

> **Cara menjalankan:** buka notebook ini di Google Colab, lalu pilih menu **Runtime → Run all**. Semua pustaka yang digunakan (pandas, numpy, matplotlib, plotly, ipywidgets, scipy) sudah tersedia di Colab.

## 1. Pendahuluan

### 1.1 Latar Belakang
Harga bahan pangan pokok seperti beras, cabai, bawang, telur, dan minyak goreng berfluktuasi setiap minggu, terutama menjelang hari besar keagamaan dan saat musim hujan. Di Provinsi Bengkulu, data harga dari sepuluh kabupaten/kota umumnya dikumpulkan dalam bentuk tabel spreadsheet mingguan. Bentuk penyajian ini menyulitkan berbagai pengguna:

- **Analis dinas** harus menyalin dan menghitung ulang data secara manual untuk mengetahui tren.
- **Pedagang dan masyarakat** kesulitan membaca tabel berisi ratusan angka.
- **Mahasiswa/jurnalis data** tidak dapat dengan cepat membandingkan harga antarwilayah.

Kuliah praktisi menekankan bahwa sistem informasi yang baik tidak hanya benar secara teknis, tetapi juga **menyajikan informasi secara jelas, mudah dipahami, dan mudah diakses**. Oleh karena itu, studi kasus ini merancang dashboard **SIPANGAN Bengkulu** (Sistem Informasi Pemantauan Harga Pangan) dengan menerapkan prinsip HCI dan visualisasi data yang dibahas dalam kuliah.

### 1.2 Rumusan Masalah
1. Bagaimana mengidentifikasi kebutuhan pengguna dashboard harga pangan dengan pendekatan *User-Centered Design*?
2. Bagaimana menerapkan prinsip psikologi kognitif (*Recognition over Recall*, Gestalt) dan prinsip *usability* dalam rancangan antarmuka?
3. Bagaimana menyajikan data harga dengan visualisasi yang tepat sesuai prinsip Tufte dan mantra Shneiderman?
4. Bagaimana mengevaluasi apakah rancangan baru lebih mudah digunakan dibandingkan penyajian lama?

### 1.3 Tujuan
1. Menyusun persona, skenario, dan daftar tugas pengguna.
2. Membuat prototipe *low-fidelity* dan *high-fidelity* (dashboard interaktif) yang menerapkan prinsip HCI.
3. Membandingkan visualisasi yang buruk dan yang baik secara terukur.
4. Melakukan evaluasi *usability* (tingkat keberhasilan tugas, waktu, kesalahan, dan skor SUS) antara desain lama (v1) dan desain baru (v2).

### 1.4 Batasan
- Data harga yang digunakan adalah **data sintetis (simulasi)** yang dibangkitkan dengan pola realistis (tren, efek hari besar, perbedaan antarwilayah). Data ini **bukan data resmi** dan hanya dipakai untuk mendemonstrasikan metode.
- Data hasil uji *usability* juga merupakan **data simulasi** untuk menunjukkan cara pengolahan dan analisis metrik *usability*.
- "Harga acuan" pada setiap komoditas adalah **nilai asumsi studi kasus**, bukan ketetapan pemerintah.

### 1.5 Metodologi
Perancangan mengikuti siklus desain berulang (*iterative design*) yang dibahas dalam kuliah:

```
 ┌──────────────────┐   ┌──────────────┐   ┌───────────────────┐   ┌──────────────┐
 │ 1. Analisis      │──▶│ 2. Konsep    │──▶│ 3. Prototipe      │──▶│ 4. Evaluasi  │
 │    kebutuhan     │   │    desain    │   │  low-fi → high-fi │   │   usability  │
 └──────────────────┘   └──────────────┘   └───────────────────┘   └──────┬───────┘
          ▲                                                                │
          └──────────────────── 5. Iterasi (perbaikan) ◀───────────────────┘
```

In [ ]:
# =============================================================
# 0. PERSIAPAN LINGKUNGAN
# =============================================================
import importlib.util, subprocess, sys
for paket in ["plotly", "ipywidgets", "scipy"]:
    if importlib.util.find_spec(paket) is None:          # instal hanya jika belum ada
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paket])

import os, difflib, logging, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.colors import TwoSlopeNorm, to_rgb
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from scipy import stats

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
pio.renderers.default = "colab"           # agar grafik plotly tampil di Google Colab
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- Gaya grafik matplotlib yang bersih (prinsip Tufte: minim ornamen) ----
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
    "axes.edgecolor": "#6B7280", "axes.labelcolor": "#374151",
    "xtick.color": "#4B5563", "ytick.color": "#4B5563",
})

# ---- Palet Okabe-Ito: aman bagi penyandang buta warna ----
OI = {"oranye": "#E69F00", "biru_langit": "#56B4E9", "hijau": "#009E73", "kuning": "#F0E442",
      "biru": "#0072B2", "merah_bata": "#D55E00", "ungu": "#CC79A7", "hitam": "#000000"}
PALET = [OI["biru"], OI["oranye"], OI["hijau"], OI["merah_bata"], OI["ungu"], OI["biru_langit"]]
ABU = "#9CA3AF"; TEKS = "#1F2937"
WARNA_STATUS = {"Normal": OI["biru"], "Waspada": OI["oranye"], "Tinggi": OI["merah_bata"]}

NAMA_BULAN = ["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"]

def rupiah(x):
    # Format angka ke gaya Indonesia, contoh 13500 -> 'Rp 13.500'
    return "Rp " + f"{x:,.0f}".replace(",", ".")

def tgl_id(t):
    t = pd.Timestamp(t)
    return f"{t.day} {NAMA_BULAN[t.month - 1]} {t.year}"

def format_bulan(ax, interval=3):
    # Label sumbu waktu dalam bahasa Indonesia, misal 'Apr 2025'
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    ax.xaxis.set_major_formatter(plt.FuncFormatter(
        lambda x, _: f"{NAMA_BULAN[mdates.num2date(x).month - 1]} {mdates.num2date(x).year}"))

def format_ribuan(ax, sumbu="y"):
    f = plt.FuncFormatter(lambda x, _: f"{x/1000:.0f} rb")
    (ax.yaxis if sumbu == "y" else ax.xaxis).set_major_formatter(f)

os.makedirs("output", exist_ok=True)
print("Lingkungan siap. pandas", pd.__version__, "| numpy", np.__version__)

## 2. Tahap 1 — Analisis Kebutuhan Pengguna (*User-Centered Design*)

Sesuai prinsip *User-Centered Design*, pengembangan dimulai dari **siapa penggunanya, apa tujuannya, dan dalam konteks apa mereka menggunakan sistem**, bukan dari keinginan pengembang. Analisis kebutuhan disusun melalui tiga artefak:

1. **Persona** — gambaran representatif kelompok pengguna.
2. **Daftar tugas pengguna (*user tasks*)** — tugas konkret yang nantinya juga dipakai sebagai skenario uji *usability*.
3. **Kebutuhan fungsional dan non-fungsional** — diturunkan langsung dari persona dan tugas.

In [ ]:
# =============================================================
# 2.1 PERSONA PENGGUNA
# =============================================================
persona = pd.DataFrame([
    {"Persona": "Bu Rina (38 th)", "Peran": "Analis Dinas Ketahanan Pangan Provinsi",
     "Literasi digital": "Menengah", "Perangkat": "Laptop kantor",
     "Tujuan": "Menyusun laporan mingguan dan mendeteksi lonjakan harga lebih awal",
     "Kendala saat ini": "Merekap 800 baris spreadsheet tiap minggu secara manual"},
    {"Persona": "Pak Darman (52 th)", "Peran": "Pedagang sayur Pasar Panorama, Kota Bengkulu",
     "Literasi digital": "Rendah", "Perangkat": "Ponsel Android",
     "Tujuan": "Mengetahui harga cabai dan bawang di wilayah lain sebelum kulakan",
     "Kendala saat ini": "Tidak paham istilah teknis, kesulitan membaca tabel angka yang panjang"},
    {"Persona": "Sari (22 th)", "Peran": "Mahasiswa / jurnalis data kampus",
     "Literasi digital": "Tinggi", "Perangkat": "Laptop & ponsel",
     "Tujuan": "Membandingkan tren harga antarwilayah untuk tulisan analisis",
     "Kendala saat ini": "Data tersebar, tidak ada grafik yang bisa difilter"},
])
display(persona)

# =============================================================
# 2.2 DAFTAR TUGAS PENGGUNA (dipakai lagi pada tahap evaluasi)
# =============================================================
tugas = pd.DataFrame([
    {"Kode": "T1", "Tugas": "Mengetahui harga rata-rata terkini suatu komoditas di Provinsi Bengkulu",
     "Persona utama": "Pak Darman", "Prinsip Shneiderman": "Overview"},
    {"Kode": "T2", "Tugas": "Melihat tren harga suatu komoditas selama 6 bulan terakhir",
     "Persona utama": "Bu Rina", "Prinsip Shneiderman": "Overview + Zoom"},
    {"Kode": "T3", "Tugas": "Membandingkan harga antarwilayah dan menemukan wilayah termahal",
     "Persona utama": "Sari", "Prinsip Shneiderman": "Filter"},
    {"Kode": "T4", "Tugas": "Mengidentifikasi komoditas yang harganya melewati batas waspada",
     "Persona utama": "Bu Rina", "Prinsip Shneiderman": "Overview"},
    {"Kode": "T5", "Tugas": "Melihat detail harga mingguan suatu wilayah pada periode tertentu",
     "Persona utama": "Sari", "Prinsip Shneiderman": "Details-on-demand"},
])
display(tugas)

In [ ]:
# =============================================================
# 2.3 KEBUTUHAN FUNGSIONAL & NON-FUNGSIONAL (turunan dari persona & tugas)
# =============================================================
kebutuhan = pd.DataFrame([
    ("KF-01", "Fungsional", "Menampilkan ringkasan harga terkini dalam bentuk kartu indikator (KPI)", "T1, T4"),
    ("KF-02", "Fungsional", "Menampilkan grafik tren harga yang dapat diperbesar (zoom) per periode", "T2"),
    ("KF-03", "Fungsional", "Membandingkan harga antarwilayah dalam grafik batang terurut", "T3"),
    ("KF-04", "Fungsional", "Menyaring data berdasarkan komoditas, wilayah, dan rentang waktu", "T2, T3, T5"),
    ("KF-05", "Fungsional", "Menampilkan tabel detail mingguan hanya ketika diminta", "T5"),
    ("KF-06", "Fungsional", "Memberi status Normal / Waspada / Tinggi terhadap harga acuan", "T4"),
    ("KNF-01", "Non-fungsional (Usability)", "Pilihan berupa menu, bukan isian teks bebas (Recognition over Recall)", "Semua"),
    ("KNF-02", "Non-fungsional (Feedback)", "Setiap perubahan filter langsung memberi umpan balik tertulis", "Semua"),
    ("KNF-03", "Non-fungsional (Aksesibilitas)", "Warna aman buta warna, kontras teks ≥ 4,5:1, status tidak hanya dibedakan warna", "Semua"),
    ("KNF-04", "Non-fungsional (Konsistensi)", "Format Rupiah, satuan, dan warna status seragam di semua komponen", "Semua"),
    ("KNF-05", "Non-fungsional (Pencegahan kesalahan)", "Batasan input: periode tidak bisa terbalik, maksimal 5 wilayah dibandingkan", "T2, T3"),
], columns=["Kode", "Jenis", "Deskripsi kebutuhan", "Mendukung tugas"])
display(kebutuhan)

## 3. Tahap 2 — Konsep Desain Berbasis Psikologi Kognitif

Tabel berikut memetakan prinsip yang dibahas dalam kuliah ke keputusan desain konkret pada SIPANGAN Bengkulu.

| Prinsip | Penjelasan singkat | Keputusan desain pada SIPANGAN |
|---|---|---|
| **Recognition over Recall** | Pengguna lebih mudah *mengenali* pilihan daripada *mengingat* perintah/nama | Komoditas & wilayah dipilih dari menu tarik-turun (*dropdown*), bukan diketik |
| **Gestalt – Kedekatan** | Elemen yang berdekatan dipersepsikan sebagai satu kelompok | Semua kontrol filter dikumpulkan dalam satu panel |
| **Gestalt – Kesamaan** | Elemen dengan warna/bentuk sama dianggap sejenis | Warna status (biru/oranye/merah bata) sama di kartu KPI, grafik batang, dan heatmap |
| **Gestalt – Wilayah bersama** | Elemen dalam satu bingkai dianggap satu kesatuan | Kartu KPI diberi bingkai; grafik dipisah dalam area masing-masing |
| **Visibilitas** | Fungsi penting harus terlihat | Filter selalu tampil di bagian atas, tidak disembunyikan di menu |
| **Umpan balik (*feedback*)** | Sistem memberi respons atas setiap aksi | Baris status "✔ Menampilkan N data…" berubah setiap filter diganti |
| **Konsistensi** | Tampilan dan istilah seragam | Format "Rp 13.500", satuan, dan istilah status yang sama di semua komponen |
| **Batasan (*constraints*)** | Mencegah kesalahan sebelum terjadi | Slider periode tidak bisa terbalik; perbandingan dibatasi 5 wilayah |
| **Aksesibilitas** | Dapat digunakan oleh semua kalangan | Palet Okabe-Ito, kontras teks diuji, status diberi label teks dan ikon |

Dua prinsip pertama dibuktikan secara terukur pada sel kode berikut.

In [ ]:
# =============================================================
# 3.1 SIMULASI RECOGNITION OVER RECALL
# Pengguna diminta mencari komoditas dengan MENGETIK (recall).
# Sistem lama hanya menerima nama yang persis sama.
# =============================================================
DAFTAR_KOMODITAS_RESMI = ["Beras Medium", "Cabai Merah Keriting", "Bawang Merah", "Bawang Putih",
                          "Daging Ayam Ras", "Telur Ayam Ras", "Minyak Goreng Curah", "Gula Pasir"]

ketikan_pengguna = ["Cabai Merah Keriting", "cabe merah keriting", "cabai keriting", "Bawang Merah",
                    "bawang mrh", "Beras medium", "beras", "Telur Ayam", "telur ayam ras",
                    "Minyak goreng", "gula pasir", "daging ayam", "Bawang putih ", "Gula Pasir",
                    "Beras Medium"]

def cari_persis(teks):                          # perilaku sistem lama (recall)
    return teks if teks in DAFTAR_KOMODITAS_RESMI else None

def saran_sistem(teks):                         # umpan balik + pemulihan kesalahan
    kandidat = difflib.get_close_matches(teks.strip().lower(),
                                         [k.lower() for k in DAFTAR_KOMODITAS_RESMI], n=1, cutoff=0.5)
    if not kandidat:
        return None
    return next(k for k in DAFTAR_KOMODITAS_RESMI if k.lower() == kandidat[0])

hasil_rr = pd.DataFrame({
    "Ketikan pengguna": ketikan_pengguna,
    "Ditemukan (sistem lama)": ["✔" if cari_persis(t) else "✘ tidak ditemukan" for t in ketikan_pengguna],
    "Saran sistem ('Mungkin maksud Anda…')": [saran_sistem(t) or "-" for t in ketikan_pengguna],
})
display(hasil_rr)

n = len(ketikan_pengguna)
berhasil_recall = sum(cari_persis(t) is not None for t in ketikan_pengguna)
berhasil_saran = sum(saran_sistem(t) is not None for t in ketikan_pengguna)
ringkas_rr = pd.DataFrame({
    "Pendekatan": ["Recall: ketik nama persis", "Recall + saran otomatis", "Recognition: pilih dari dropdown"],
    "Tingkat keberhasilan (%)": [100 * berhasil_recall / n, 100 * berhasil_saran / n, 100.0],
})
display(ringkas_rr.round(1))
print(f"Hanya {berhasil_recall} dari {n} ketikan ({100*berhasil_recall/n:.0f}%) yang dikenali sistem lama.")
print("Dengan dropdown, pengguna cukup MENGENALI pilihan sehingga salah ketik tidak mungkin terjadi.")

In [ ]:
# =============================================================
# 3.2 DEMONSTRASI PRINSIP GESTALT
# =============================================================
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
rng_demo = np.random.default_rng(1)

# (a) Kedekatan: 12 titik, jarak antarkelompok lebih lebar -> terlihat 3 kelompok
xs = np.concatenate([np.arange(4) * 0.6 + off for off in (0, 3.6, 7.2)])
for y in (0, 0.6, 1.2):
    ax[0].scatter(xs, np.full_like(xs, y), s=90, color=OI["biru"])
ax[0].set_title("(a) Kedekatan")

# (b) Kesamaan: grid seragam, warna/bentuk berselang -> terlihat kolom
for i in range(6):
    for j in range(5):
        if i % 2 == 0:
            ax[1].scatter(i, j, s=110, color=OI["biru"], marker="o")
        else:
            ax[1].scatter(i, j, s=110, color=OI["oranye"], marker="s")
ax[1].set_title("(b) Kesamaan")

# (c) Wilayah bersama: titik berjarak sama, bingkai membentuk kelompok
xs_c = np.arange(9) * 1.0
ax[2].scatter(xs_c, np.zeros(9), s=90, color=OI["hijau"], zorder=3)
for start in (-0.4, 2.6, 5.6):
    ax[2].add_patch(FancyBboxPatch((start, -0.45), 2.8, 0.9, boxstyle="round,pad=0.05",
                                   fill=True, facecolor="#F3F4F6", edgecolor="#6B7280"))
ax[2].set_ylim(-1.5, 1.5)
ax[2].set_title("(c) Wilayah bersama")

# (d) Penerapan pada tata letak dashboard
a = ax[3]
a.set_xlim(0, 10); a.set_ylim(0, 7)
a.add_patch(FancyBboxPatch((0.2, 5.4), 9.6, 1.3, boxstyle="round,pad=0.05", facecolor="#EFF6FF", edgecolor=OI["biru"]))
a.text(0.5, 6.05, "Panel Filter (kedekatan)", fontsize=8, color=TEKS, va="center")
for k in range(4):
    a.add_patch(FancyBboxPatch((0.2 + k * 2.45, 3.7), 2.2, 1.3, boxstyle="round,pad=0.05",
                               facecolor="white", edgecolor=WARNA_STATUS["Waspada"] if k == 3 else "#6B7280"))
a.text(0.4, 3.35, "Kartu KPI (wilayah bersama)", fontsize=8, color=TEKS)
a.add_patch(Rectangle((0.2, 0.3), 6.0, 2.7, facecolor="white", edgecolor="#6B7280"))
a.add_patch(Rectangle((6.5, 0.3), 3.3, 2.7, facecolor="white", edgecolor="#6B7280"))
a.plot([0.6, 2, 3.2, 4.4, 5.8], [0.9, 1.6, 1.2, 2.4, 2.0], color=OI["biru"])
for i, h in enumerate([0.8, 1.3, 1.9, 2.3]):
    a.add_patch(Rectangle((6.8, 0.5 + i * 0.6), h, 0.4,
                          color=list(WARNA_STATUS.values())[min(i, 2)]))
a.set_title("(d) Penerapan di dashboard")

for axis in ax:
    axis.set_xticks([]); axis.set_yticks([])
    for s in axis.spines.values():
        s.set_visible(False)
plt.suptitle("Prinsip Gestalt: otak mengelompokkan elemen berdasarkan jarak, kemiripan, dan bingkai",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Tahap 3 — Penyiapan Data

### 4.1 Pembangkitan data sintetis
Data harga mingguan dibangkitkan untuk **8 komoditas × 10 kabupaten/kota × 91 minggu** (Januari 2025 – September 2026). Pola yang dimodelkan:

- **Tren** kenaikan harga tahunan yang berbeda per komoditas.
- **Efek hari besar**: lonjakan menjelang Idulfitri 2025, Natal 2025, dan Idulfitri 2026.
- **Guncangan pasokan** (proses AR(1)) yang tinggi pada komoditas hortikultura seperti cabai dan bawang merah.
- **Perbedaan wilayah**: wilayah sentra produksi sayur (Rejang Lebong, Kepahiang) lebih murah untuk hortikultura, sedangkan wilayah yang jauh (Kaur, Mukomuko) lebih mahal.

Agar mirip kondisi nyata, data mentah sengaja diberi **masalah kualitas data**: nilai kosong, angka nol, salah ketik (kelebihan satu digit nol), format teks "Rp 13.500", nama wilayah/komoditas tidak seragam, dan baris ganda.

In [ ]:
# =============================================================
# 4.1 PEMBANGKITAN DATA HARGA (SINTETIS)
# =============================================================
rng = np.random.default_rng(2026)   # seed tetap agar hasil dapat direproduksi

KOMODITAS = {   # dasar = harga awal (Rp), vol = volatilitas mingguan, musim = efek hari besar,
                # tren = kenaikan per tahun, acuan = harga acuan (ASUMSI studi kasus)
    "Beras Medium":         dict(satuan="kg",    dasar=13200, vol=0.005, musim=0.02, tren=0.030, acuan=13900),
    "Cabai Merah Keriting": dict(satuan="kg",    dasar=42000, vol=0.060, musim=0.25, tren=0.020, acuan=47000),
    "Bawang Merah":         dict(satuan="kg",    dasar=36000, vol=0.040, musim=0.15, tren=0.020, acuan=40000),
    "Bawang Putih":         dict(satuan="kg",    dasar=37000, vol=0.020, musim=0.06, tren=0.030, acuan=40000),
    "Daging Ayam Ras":      dict(satuan="kg",    dasar=35000, vol=0.015, musim=0.10, tren=0.030, acuan=38500),
    "Telur Ayam Ras":       dict(satuan="kg",    dasar=28500, vol=0.015, musim=0.08, tren=0.030, acuan=31000),
    "Minyak Goreng Curah":  dict(satuan="liter", dasar=16500, vol=0.008, musim=0.03, tren=0.040, acuan=17600),
    "Gula Pasir":           dict(satuan="kg",    dasar=17800, vol=0.006, musim=0.03, tren=0.030, acuan=18500),
}
WILAYAH = {     # faktor = selisih harga umum, horti = faktor khusus komoditas hortikultura
    "Kota Bengkulu":    dict(faktor=1.00, horti=1.00),
    "Bengkulu Tengah":  dict(faktor=1.01, horti=1.00),
    "Seluma":           dict(faktor=1.02, horti=1.00),
    "Bengkulu Selatan": dict(faktor=1.03, horti=1.01),
    "Kaur":             dict(faktor=1.07, horti=1.03),
    "Bengkulu Utara":   dict(faktor=1.02, horti=1.00),
    "Mukomuko":         dict(faktor=1.06, horti=1.03),
    "Rejang Lebong":    dict(faktor=0.99, horti=0.90),
    "Kepahiang":        dict(faktor=1.00, horti=0.94),
    "Lebong":           dict(faktor=1.04, horti=0.97),
}
HORTIKULTURA = {"Cabai Merah Keriting", "Bawang Merah"}

TANGGAL = pd.date_range("2025-01-06", "2026-09-28", freq="W-MON")
t_tahun = np.asarray((TANGGAL - TANGGAL[0]).days) / 365.25
HARI_BESAR = pd.to_datetime(["2025-03-31", "2025-12-25", "2026-03-20"])  # Idulfitri 2025, Natal 2025, Idulfitri 2026

def efek_hari_besar(tgl):
    efek = np.zeros(len(tgl))
    for hb in HARI_BESAR:
        minggu = np.asarray((tgl - hb).days) / 7
        efek += np.exp(-0.5 * ((minggu + 1) / 2.0) ** 2)      # puncak ±1 minggu sebelum hari besar
    return efek

def guncangan_ar1(n, vol, rho=0.85):
    g = np.zeros(n)
    for i in range(1, n):
        g[i] = rho * g[i - 1] + rng.normal(0, vol)
    return np.clip(g, -0.30, 0.70)

EFEK = efek_hari_besar(TANGGAL)
potongan = []
for kom, p in KOMODITAS.items():
    pola = p["dasar"] * (1 + p["tren"] * t_tahun) * (1 + p["musim"] * EFEK) * (1 + guncangan_ar1(len(TANGGAL), p["vol"]))
    for wil, w in WILAYAH.items():
        faktor = w["faktor"] * (w["horti"] if kom in HORTIKULTURA else 1.0)
        harga = pola * faktor * (1 + rng.normal(0, 0.015, len(TANGGAL)))
        potongan.append(pd.DataFrame({"tanggal": TANGGAL, "wilayah": wil, "komoditas": kom,
                                      "satuan": p["satuan"], "harga": np.round(harga / 50) * 50}))
data_ideal = pd.concat(potongan, ignore_index=True)

# ---- Menyuntikkan masalah kualitas data agar menyerupai data lapangan ----
df_mentah = data_ideal.copy()
df_mentah["harga"] = df_mentah["harga"].astype(object)
acak = rng.permutation(len(df_mentah))
n = len(df_mentah)
k_kosong, k_nol, k_typo, k_teks, k_nama = int(n * .020), int(n * .004), int(n * .004), int(n * .030), int(n * .030)
i_kosong = acak[:k_kosong]
i_nol    = acak[k_kosong:k_kosong + k_nol]
i_typo   = acak[k_kosong + k_nol:k_kosong + k_nol + k_typo]
i_teks   = acak[k_kosong + k_nol + k_typo:k_kosong + k_nol + k_typo + k_teks]
i_nama   = rng.choice(n, k_nama, replace=False)

df_mentah.loc[i_kosong, "harga"] = np.nan
df_mentah.loc[i_nol, "harga"] = 0
df_mentah.loc[i_typo, "harga"] = [h * 10 for h in df_mentah.loc[i_typo, "harga"]]
df_mentah.loc[i_teks, "harga"] = [rupiah(h).replace("Rp ", "Rp") for h in df_mentah.loc[i_teks, "harga"]]
df_mentah.loc[i_nama, "wilayah"] = [w.lower() + " " for w in df_mentah.loc[i_nama, "wilayah"]]
df_mentah.loc[i_nama[::2], "komoditas"] = [k.upper() for k in df_mentah.loc[i_nama[::2], "komoditas"]]
duplikat = df_mentah.sample(frac=0.005, random_state=7)
df_mentah = pd.concat([df_mentah, duplikat]).sample(frac=1, random_state=7).reset_index(drop=True)

print(f"Data mentah: {len(df_mentah):,} baris × {df_mentah.shape[1]} kolom".replace(",", "."))
display(df_mentah.head(10))

In [ ]:
# =============================================================
# 4.2 AUDIT KUALITAS DATA MENTAH
# =============================================================
def ke_angka(x):
    if isinstance(x, str):
        x = x.replace("Rp", "").replace(".", "").replace(",", ".").strip()
    return pd.to_numeric(x, errors="coerce")

harga_num = df_mentah["harga"].map(ke_angka).astype(float)
median_grup = harga_num.groupby([df_mentah["wilayah"].str.strip().str.lower(),
                                 df_mentah["komoditas"].str.lower()]).transform("median")
audit = pd.DataFrame([
    ("Harga kosong (NaN)", int(df_mentah["harga"].isna().sum()), "Interpolasi linear per wilayah-komoditas"),
    ("Harga berformat teks 'Rp…'", int(df_mentah["harga"].map(lambda v: isinstance(v, str)).sum()), "Dikonversi ke angka"),
    ("Harga nol / negatif", int((harga_num <= 0).sum()), "Dianggap tidak valid → interpolasi"),
    ("Harga tidak wajar (> 3× median)", int((harga_num > 3 * median_grup).sum()), "Dianggap salah ketik → interpolasi"),
    ("Nama wilayah/komoditas tidak seragam",
     int((~df_mentah["wilayah"].isin(WILAYAH) | ~df_mentah["komoditas"].isin(KOMODITAS)).sum()),
     "Distandarkan ke nama baku"),
    ("Baris ganda (setelah nama distandarkan)", None, "Dihapus, disimpan satu"),
], columns=["Masalah", "Jumlah baris", "Penanganan"])

kunci = pd.DataFrame({"t": df_mentah["tanggal"],
                      "w": df_mentah["wilayah"].str.strip().str.lower(),
                      "k": df_mentah["komoditas"].str.lower()})
audit.loc[5, "Jumlah baris"] = int(kunci.duplicated().sum())
display(audit)

In [ ]:
# =============================================================
# 4.3 PEMBERSIHAN DATA (DATA CLEANING PIPELINE)
# =============================================================
def bersihkan(mentah):
    d = mentah.copy()
    peta_wil = {w.lower(): w for w in WILAYAH}
    peta_kom = {k.lower(): k for k in KOMODITAS}
    d["wilayah"] = d["wilayah"].str.strip().str.lower().map(peta_wil)          # 1. standarisasi nama
    d["komoditas"] = d["komoditas"].str.strip().str.lower().map(peta_kom)
    d["harga"] = d["harga"].map(ke_angka).astype(float)                         # 2. teks -> angka
    d = d.drop_duplicates(subset=["tanggal", "wilayah", "komoditas"])           # 3. hapus duplikat
    med = d.groupby(["wilayah", "komoditas"])["harga"].transform("median")
    tidak_wajar = (d["harga"] <= 0) | (d["harga"] > 3 * med) | (d["harga"] < med / 3)
    d.loc[tidak_wajar, "harga"] = np.nan                                        # 4. tandai outlier
    d = d.sort_values(["komoditas", "wilayah", "tanggal"])
    d["harga"] = (d.groupby(["wilayah", "komoditas"])["harga"]                 # 5. interpolasi
                   .transform(lambda s: s.interpolate(limit_direction="both")).round(-1))
    return d.reset_index(drop=True)

df = bersihkan(df_mentah)

# Tambahan atribut turunan: harga acuan dan status
df["acuan"] = df["komoditas"].map({k: v["acuan"] for k, v in KOMODITAS.items()})
df["rasio_acuan"] = df["harga"] / df["acuan"]
df["status"] = np.select([df["rasio_acuan"] <= 1.00, df["rasio_acuan"] <= 1.10],
                         ["Normal", "Waspada"], default="Tinggi")

# ---- Validasi hasil pembersihan ----
validasi = pd.DataFrame({
    "Pemeriksaan": ["Jumlah baris (harus 8 × 10 × 91 = 7.280)", "Nilai kosong tersisa",
                    "Nama wilayah unik", "Nama komoditas unik",
                    "Rata-rata selisih absolut terhadap data ideal (%)"],
    "Hasil": [len(df), int(df["harga"].isna().sum()), df["wilayah"].nunique(), df["komoditas"].nunique(),
              round((df.merge(data_ideal, on=["tanggal", "wilayah", "komoditas"], suffixes=("", "_ideal"))
                       .eval("abs(harga - harga_ideal) / harga_ideal").mean() * 100), 2)],
})
display(validasi)
display(df.sample(6, random_state=3))

## 5. Tahap 4 — Prototipe *Low-Fidelity* (Wireframe)

Prototipe *low-fidelity* berupa sketsa kasar untuk menguji **struktur dan alur informasi** sebelum mengerjakan tampilan akhir. Sketsa ini sengaja dibuat dengan gaya "coretan tangan" (`plt.xkcd()`) agar fokus diskusi tetap pada tata letak, bukan warna atau estetika.

Tata letak disusun mengikuti mantra Shneiderman: **(1) Overview** di bagian atas, **(2) Zoom & Filter** pada panel kiri dan grafik, **(3) Details-on-demand** di bagian bawah.

In [ ]:
# =============================================================
# 5. WIREFRAME LOW-FIDELITY
# =============================================================
with plt.xkcd(scale=0.6, length=120, randomness=2):
    fig, ax = plt.subplots(figsize=(14, 8.5))
    ax.set_xlim(0, 100); ax.set_ylim(0, 62); ax.axis("off")

    def kotak(x, y, w, h, label="", fs=10, isi="white"):
        ax.add_patch(Rectangle((x, y), w, h, fill=True, facecolor=isi, edgecolor="black", lw=1.3))
        if label:
            ax.text(x + 1, y + h - 1.2, label, fontsize=fs, va="top")

    kotak(0, 56, 100, 6, "SIPANGAN Bengkulu  -  Pemantauan Harga Pangan          Diperbarui: [tanggal]", fs=12, isi="#F3F4F6")
    # Panel filter (kiri)
    kotak(0, 14, 22, 41, "FILTER", fs=11, isi="#F9FAFB")
    for i, teks in enumerate(["Komoditas  [Cabai Merah  v]", "Wilayah  [ ] Kota Bengkulu\n             [ ] Rejang Lebong\n             [ ] ...",
                              "Periode  |==o-------o==|", "[ Reset filter ]"]):
        ax.text(1.5, 50 - i * 8.5, teks, fontsize=9, va="top")
    ax.text(1.5, 17.5, "Status: 'Menampilkan\n728 data...'", fontsize=8, va="top", style="italic")
    # KPI
    for i, judul in enumerate(["Harga rata-rata\nRp 45.000/kg", "Perubahan 4 mgg\n+ 6,2 % (naik)",
                               "Wilayah termahal\nKaur", "Status\n! WASPADA"]):
        kotak(24 + i * 19, 45, 17.5, 10, judul, fs=9)
    # Grafik tren
    kotak(24, 14, 46, 29, "Tren harga mingguan (grafik garis)", fs=9)
    xx = np.linspace(27, 67, 30)
    ax.plot(xx, 24 + 6 * np.sin(xx / 4) + (xx - 27) / 4, color="black")
    ax.plot([27, 67], [33, 33], ls="--", color="black"); ax.text(55, 34, "harga acuan", fontsize=8)
    ax.text(27, 16, "[3 bln] [6 bln] [Semua]   <== zoom", fontsize=8)
    # Grafik batang
    kotak(72, 14, 28, 29, "Perbandingan wilayah\n(batang terurut)", fs=9)
    for i, w in enumerate([20, 17, 14, 12, 10, 8]):
        ax.add_patch(Rectangle((74, 17 + i * 3.6), w, 2.4, facecolor="#D1D5DB", edgecolor="black"))
    # Detail
    kotak(0, 0, 100, 12.5, "> Tampilkan tabel detail mingguan   (muncul hanya jika diminta)", fs=9)
    for j in range(4):
        ax.plot([2, 98], [2 + j * 2.4, 2 + j * 2.4], color="#9CA3AF", lw=0.8)

    # Anotasi mantra Shneiderman
    ax.annotate("(1) OVERVIEW FIRST", xy=(62, 55), xytext=(104, 50), fontsize=11, color=OI["merah_bata"],
                arrowprops=dict(arrowstyle="->", color=OI["merah_bata"]), annotation_clip=False)
    ax.annotate("(2) ZOOM & FILTER", xy=(0, 36), xytext=(-24, 36), fontsize=11, color=OI["merah_bata"],
                arrowprops=dict(arrowstyle="->", color=OI["merah_bata"]), annotation_clip=False)
    ax.annotate("(3) DETAILS-ON-DEMAND", xy=(50, 6), xytext=(70, -8), fontsize=11, color=OI["merah_bata"],
                arrowprops=dict(arrowstyle="->", color=OI["merah_bata"]), annotation_clip=False)
    plt.title("Wireframe Low-Fidelity SIPANGAN Bengkulu", fontsize=14, pad=14, loc="center")
    plt.savefig("output/wireframe_lowfi.png", bbox_inches="tight")
    plt.show()

## 6. Tahap 5 — Prinsip Visualisasi Data

### 6.1 Prinsip Tufte: hindari *chartjunk* dan distorsi
Edward Tufte menyarankan memaksimalkan **data-ink** (tinta yang benar-benar menyampaikan data) dan menghindari hiasan yang tidak menambah informasi. Tufte juga memperkenalkan **Lie Factor**:

> Lie Factor = (besar efek yang tampak pada grafik) ÷ (besar efek pada data). Nilai ideal = 1.

Sel berikut membandingkan grafik "buruk" (warna pelangi, latar gradasi, garis tebal, sumbu terpotong, angka desimal berlebihan, urutan acak) dengan grafik "baik" untuk data yang **sama persis**.

In [ ]:
# =============================================================
# 6.1 GRAFIK BURUK vs GRAFIK BAIK (PRINSIP TUFTE)
# =============================================================
KOM_DEMO = "Bawang Merah"
akhir = df["tanggal"].max()
s = df[(df["komoditas"] == KOM_DEMO) & (df["tanggal"] == akhir)].set_index("wilayah")["harga"]
acuan_demo = KOMODITAS[KOM_DEMO]["acuan"]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5.6), gridspec_kw={"width_ratios": [1, 1.1]})

# ---------------- (A) GRAFIK BURUK ----------------
y0 = s.min() * 0.97                                    # sumbu Y sengaja dipotong
warna_pelangi = plt.cm.jet(np.linspace(0, 1, len(s)))
a1.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=(-0.7, len(s) - 0.3, y0, s.max() * 1.02),
          aspect="auto", cmap="YlOrRd", alpha=0.35, zorder=0)
a1.bar(np.arange(len(s)) + 0.12, s.values - y0, bottom=y0, color="gray", alpha=0.5, zorder=1)  # bayangan 3D palsu
batang = a1.bar(np.arange(len(s)), s.values - y0, bottom=y0, color=warna_pelangi, edgecolor="black",
                lw=2, hatch="//", zorder=2)
for i, v in enumerate(s.values):
    a1.text(i, v + 50, f"{v:.2f}", rotation=90, ha="center", fontsize=8, fontweight="bold")
a1.set_xticks(np.arange(len(s)), s.index, rotation=45, ha="right")
a1.set_ylim(y0, s.max() * 1.02)
a1.grid(True, color="black", lw=1.2)
a1.legend(batang, s.index, fontsize=6, ncol=2, loc="upper left")
for sp in a1.spines.values():
    sp.set_visible(True); sp.set_linewidth(3)
a1.set_title(f"(A) GRAFIK HARGA {KOM_DEMO.upper()}!!!", color="red", fontsize=14)

# ---------------- (B) GRAFIK BAIK ----------------
s_urut = s.sort_values()
warna = [WARNA_STATUS["Tinggi"] if v > acuan_demo * 1.10 else WARNA_STATUS["Waspada"] if v > acuan_demo
         else "#9CA3AF" for v in s_urut.values]
a2.barh(s_urut.index, s_urut.values, color=warna, height=0.65)
for i, v in enumerate(s_urut.values):
    a2.text(v + s_urut.max() * 0.01, i, rupiah(v), va="center", fontsize=9, color=TEKS)
a2.axvline(acuan_demo, color=TEKS, ls="--", lw=1)
a2.text(acuan_demo, len(s_urut) - 0.35, f" harga acuan {rupiah(acuan_demo)}", fontsize=8.5, color=TEKS, va="bottom")
a2.set_xlim(0, max(s_urut.max(), acuan_demo) * 1.25)
a2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f} rb"))
a2.spines["left"].set_visible(False); a2.tick_params(axis="y", length=0)
a2.set_title(f"(B) Harga {KOM_DEMO} per wilayah, {tgl_id(akhir)} (Rp/kg)")
a2.text(0, -2.3, "Abu-abu = di bawah acuan  |  Oranye = Waspada (≤10% di atas)  |  Merah bata = Tinggi (>10% di atas)",
        fontsize=8.5, color="#4B5563")
plt.tight_layout()
plt.savefig("output/tufte_buruk_vs_baik.png", bbox_inches="tight")
plt.show()

# ---- Lie Factor grafik (A) ----
efek_data = (s.max() - s.min()) / s.min()
efek_grafik = ((s.max() - y0) - (s.min() - y0)) / (s.min() - y0)
print(f"Selisih harga tertinggi vs terendah pada DATA   : {efek_data*100:6.1f} %")
print(f"Selisih tinggi batang yang TAMPAK pada grafik (A): {efek_grafik*100:6.1f} %")
print(f"Lie Factor grafik (A) = {efek_grafik/efek_data:.1f}  → perbedaan dilebih-lebihkan {efek_grafik/efek_data:.0f}×")
print("Lie Factor grafik (B) = 1,0  → sumbu dimulai dari nol, proporsi batang sesuai data")

**Perbaikan yang dilakukan pada grafik (B):**

| Masalah pada (A) | Perbaikan pada (B) | Prinsip |
|---|---|---|
| Sumbu Y dipotong sehingga selisih kecil terlihat sangat besar | Sumbu dimulai dari nol (Lie Factor = 1) | Integritas grafis (Tufte) |
| Warna pelangi tanpa makna, legenda mengulang label | Warna hanya untuk status terhadap acuan | *Data-ink ratio* |
| Latar gradasi, arsiran, bayangan 3D, garis tepi tebal | Dihapus seluruhnya | Hindari *chartjunk* |
| Label miring dan angka 2 desimal | Batang horizontal, label "Rp 45.000" | Keterbacaan |
| Urutan acak | Diurutkan dari termurah ke termahal | Memudahkan perbandingan |
| Judul tidak informatif | Judul menyebut apa, di mana, kapan, dan satuannya | Kejelasan |

In [ ]:
# =============================================================
# 6.2 MEMILIH JENIS GRAFIK YANG TEPAT
#     Garis -> perubahan dari waktu ke waktu | Batang -> perbandingan kategori
# =============================================================
kom4 = ["Beras Medium", "Cabai Merah Keriting", "Bawang Merah", "Telur Ayam Ras"]
prov = df.groupby(["tanggal", "komoditas"])["harga"].mean().unstack()
indeks = prov[kom4] / prov[kom4].iloc[0] * 100                     # indeks harga (minggu pertama = 100)
rata_wil = df[(df["komoditas"] == "Bawang Merah") & (df["tanggal"] >= akhir - pd.Timedelta(weeks=12))] \
             .groupby("wilayah")["harga"].mean().sort_values()

fig, ax = plt.subplots(2, 2, figsize=(15, 8.5))
# Kurang tepat 1: grafik batang untuk 91 titik waktu
ax[0, 0].bar(prov.index, prov["Cabai Merah Keriting"], width=5, color=ABU)
format_bulan(ax[0, 0], 4); format_ribuan(ax[0, 0])
ax[0, 0].set_title("✘ Kurang tepat: batang untuk data deret waktu (pola sulit dilihat)", color=OI["merah_bata"])
# Kurang tepat 2: grafik garis untuk kategori tanpa urutan
ax[0, 1].plot(rata_wil.sort_index().index, rata_wil.sort_index().values, marker="o", color=ABU)
ax[0, 1].tick_params(axis="x", rotation=45); format_ribuan(ax[0, 1])
ax[0, 1].set_title("✘ Kurang tepat: garis menghubungkan kategori (seolah ada urutan)", color=OI["merah_bata"])
# Tepat 1: grafik garis untuk tren
for i, k in enumerate(kom4):
    ax[1, 0].plot(indeks.index, indeks[k], color=PALET[i], lw=2)
    ax[1, 0].text(indeks.index[-1] + pd.Timedelta(days=5), indeks[k].iloc[-1], k, color=PALET[i],
                  fontsize=8.5, va="center")                       # label langsung, tanpa legenda
ax[1, 0].axhline(100, color=ABU, lw=0.8, ls=":")
for hb in HARI_BESAR:
    ax[1, 0].axvline(hb, color=ABU, lw=0.8, ls="--")
ax[1, 0].text(HARI_BESAR[0], ax[1, 0].get_ylim()[1], " Idulfitri", fontsize=8, color="#4B5563", va="top")
ax[1, 0].set_xlim(indeks.index[0], indeks.index[-1] + pd.Timedelta(days=120))
ax[1, 0].set_ylabel("Indeks harga (Jan 2025 = 100)"); format_bulan(ax[1, 0], 4)
ax[1, 0].set_title("✔ Tepat: grafik garis untuk perubahan dari waktu ke waktu", color=OI["hijau"])
# Tepat 2: grafik batang untuk perbandingan kategori
ax[1, 1].barh(rata_wil.index, rata_wil.values, color=OI["biru"], height=0.6)
for i, v in enumerate(rata_wil.values):
    ax[1, 1].text(v + 300, i, rupiah(v), va="center", fontsize=8.5)
ax[1, 1].set_xlim(0, rata_wil.max() * 1.25)
ax[1, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1000:.0f} rb"))
ax[1, 1].set_title("✔ Tepat: batang terurut untuk membandingkan wilayah\n   (rata-rata Bawang Merah 12 minggu terakhir)",
                   color=OI["hijau"])
plt.tight_layout()
plt.savefig("output/jenis_grafik.png", bbox_inches="tight")
plt.show()

### 6.3 Aksesibilitas Warna
Sekitar 8% laki-laki memiliki defisiensi penglihatan warna (buta warna merah-hijau). Dua pengujian dilakukan:

1. **Rasio kontras teks** sesuai rumus WCAG 2.1. Teks normal minimal **4,5 : 1** (level AA); teks besar minimal **3 : 1**.
2. **Simulasi buta warna** (model Machado dkk., 2009) untuk membandingkan warna status "lampu lalu lintas" (hijau–kuning–merah) yang umum dipakai dengan warna status SIPANGAN (biru–oranye–merah bata dari palet Okabe-Ito). Keterbedaan warna diukur dengan **ΔE (CIE76)**; semakin besar ΔE terkecil antarpasangan warna, semakin mudah warna dibedakan.

In [ ]:
# =============================================================
# 6.3 UJI AKSESIBILITAS: KONTRAS WCAG & SIMULASI BUTA WARNA
# =============================================================
def ke_linear(c):
    c = np.asarray(c, dtype=float)
    return np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)

def ke_srgb(c):
    c = np.clip(c, 0, 1)
    return np.where(c <= 0.0031308, 12.92 * c, 1.055 * c ** (1 / 2.4) - 0.055)

def luminansi(hex_):
    r, g, b = ke_linear(to_rgb(hex_))
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

def rasio_kontras(depan, latar):
    l1, l2 = sorted([luminansi(depan), luminansi(latar)], reverse=True)
    return (l1 + 0.05) / (l2 + 0.05)

pasangan = [("Teks utama", TEKS, "#FFFFFF"), ("Teks sekunder", "#4B5563", "#FFFFFF"),
            ("Teks abu-abu muda", "#9CA3AF", "#FFFFFF"), ("Teks kuning", OI["kuning"], "#FFFFFF"),
            ("Putih di atas biru", "#FFFFFF", OI["biru"]), ("Putih di atas merah bata", "#FFFFFF", OI["merah_bata"]),
            ("Label status Waspada", "#7A3E00", "#FFF4E5"), ("Label status Tinggi", "#8A1C1C", "#FDECEA"),
            ("Label status Normal", "#0B3D66", "#E8F1FA")]
kontras = pd.DataFrame([(n_, d_, l_, rasio_kontras(d_, l_)) for n_, d_, l_ in pasangan],
                       columns=["Elemen", "Warna teks", "Warna latar", "Rasio kontras"])
kontras["AA teks normal (≥4,5)"] = np.where(kontras["Rasio kontras"] >= 4.5, "✔ Lulus", "✘ Gagal")
kontras["AA teks besar (≥3)"] = np.where(kontras["Rasio kontras"] >= 3.0, "✔ Lulus", "✘ Gagal")
kontras["Rasio kontras"] = kontras["Rasio kontras"].round(2)
display(kontras)

# ---- Simulasi buta warna (Machado, Oliveira & Fernandes, 2009; severity = 1) ----
MATRIKS_CVD = {
    "Normal": np.eye(3),
    "Protanopia": np.array([[0.152286, 1.052583, -0.204868], [0.114503, 0.786281, 0.099216], [-0.003882, -0.048116, 1.051998]]),
    "Deuteranopia": np.array([[0.367322, 0.860646, -0.227968], [0.280085, 0.672501, 0.047413], [-0.011820, 0.042940, 0.968881]]),
}
def simulasikan(rgb, jenis):
    return ke_srgb(ke_linear(rgb) @ MATRIKS_CVD[jenis].T)

def ke_lab(rgb):
    lin = ke_linear(rgb)
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    xyz = lin @ M.T / np.array([0.95047, 1.0, 1.08883])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)
    return np.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], axis=1)

def delta_e_min(rgb):
    lab = ke_lab(rgb)
    jarak = np.linalg.norm(lab[:, None, :] - lab[None, :, :], axis=2)
    return jarak[np.triu_indices(len(rgb), 1)].min()

palet_uji = {"Lampu lalu lintas": np.array([to_rgb(c) for c in ["#2CA02C", "#FFBF00", "#D62728"]]),
             "Status SIPANGAN": np.array([to_rgb(WARNA_STATUS[s]) for s in ["Normal", "Waspada", "Tinggi"]])}

fig, ax = plt.subplots(len(palet_uji), 3, figsize=(13, 3.6))
hasil_de = []
for i, (nama_p, rgb) in enumerate(palet_uji.items()):
    for j, jenis in enumerate(MATRIKS_CVD):
        sim = simulasikan(rgb, jenis)
        ax[i, j].imshow(sim[None, :, :], aspect="auto")
        ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        de = delta_e_min(sim)
        hasil_de.append((nama_p, jenis, round(de, 1)))
        ax[i, j].set_title(f"{nama_p} – {jenis}\nΔE terkecil = {de:.1f}", fontsize=9, loc="center")
        for k_, lbl in enumerate(["Normal", "Waspada", "Tinggi"]):
            ax[i, j].text(k_, 0, lbl, ha="center", va="center", fontsize=8, color="white", fontweight="bold")
plt.suptitle("Simulasi tampilan warna status bagi penyandang buta warna", fontweight="bold")
plt.tight_layout(); plt.show()

tabel_de = pd.DataFrame(hasil_de, columns=["Palet", "Kondisi penglihatan", "ΔE terkecil"]) \
             .pivot(index="Palet", columns="Kondisi penglihatan", values="ΔE terkecil")[list(MATRIKS_CVD)]
display(tabel_de)
for nama_p in tabel_de.index:
    turun = 100 * (1 - tabel_de.loc[nama_p, "Deuteranopia"] / tabel_de.loc[nama_p, "Normal"])
    print(f"{nama_p:18s}: ΔE terkecil turun {turun:.0f}% pada deuteranopia "
          f"({tabel_de.loc[nama_p, 'Normal']:.1f} → {tabel_de.loc[nama_p, 'Deuteranopia']:.1f})")
terbaik = tabel_de["Deuteranopia"].idxmax()
print(f"\nPada kondisi deuteranopia, palet '{terbaik}' tetap paling mudah dibedakan.")
print("Palet lampu lalu lintas tampak sangat kontras bagi mata normal, tetapi hijau dan merahnya nyaris sama")
print("bagi penyandang buta warna merah-hijau. Selain itu, status di SIPANGAN selalu disertai label teks/ikon")
print("sehingga informasi tidak bergantung pada warna saja.")

## 7. Tahap 6 — Prototipe *High-Fidelity*: Dashboard Interaktif

Dashboard dibangun dengan **plotly** (grafik interaktif dengan *hover*, *zoom*, dan *range slider*) dan **ipywidgets** (kontrol filter). Struktur mengikuti mantra Shneiderman:

| Tahap Shneiderman | Implementasi |
|---|---|
| **Overview first** | Heatmap status seluruh komoditas × wilayah (7.1) dan kartu KPI di bagian atas dashboard |
| **Zoom and filter** | Dropdown komoditas, pilihan wilayah, slider periode, tombol 3 bln / 6 bln / 1 thn, *range slider* |
| **Details-on-demand** | Informasi muncul saat kursor diarahkan ke grafik (*hover*) dan tabel detail mingguan yang muncul hanya bila dicentang |

In [ ]:
# =============================================================
# 7.1 OVERVIEW FIRST: HEATMAP STATUS HARGA SELURUH KOMODITAS × WILAYAH
# =============================================================
terkini = df[df["tanggal"] == df["tanggal"].max()]
matriks = (terkini.pivot(index="komoditas", columns="wilayah", values="rasio_acuan") - 1) * 100
matriks = matriks.loc[list(KOMODITAS), sorted(WILAYAH)]

fig, ax = plt.subplots(figsize=(14, 5.2))
batas = max(abs(matriks.values.min()), abs(matriks.values.max()))
im = ax.imshow(matriks.values, cmap="PuOr_r", norm=TwoSlopeNorm(vcenter=0, vmin=-batas, vmax=batas), aspect="auto")
for i in range(matriks.shape[0]):
    for j in range(matriks.shape[1]):
        v = matriks.iat[i, j]
        simbol = "▲▲" if v > 10 else "▲" if v > 0 else ""
        ax.text(j, i, f"{v:+.1f}%{simbol}".replace(".", ","), ha="center", va="center", fontsize=8.5,
                color="white" if abs(v) > batas * 0.55 else TEKS)
ax.set_xticks(range(matriks.shape[1]), matriks.columns, rotation=30, ha="right")
ax.set_yticks(range(matriks.shape[0]), [f"{k} (/{KOMODITAS[k]['satuan']})" for k in matriks.index])
ax.set_title(f"Overview: selisih harga terhadap harga acuan, {tgl_id(df['tanggal'].max())}   "
             "(▲ = Waspada, ▲▲ = Tinggi)")
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.set_label("Selisih terhadap acuan (%)")
for s_ in ax.spines.values():
    s_.set_visible(False)
plt.tight_layout(); plt.savefig("output/overview_heatmap.png", bbox_inches="tight"); plt.show()

rekap = terkini["status"].value_counts().reindex(["Normal", "Waspada", "Tinggi"], fill_value=0)
print("Rekap status (komoditas × wilayah) minggu terakhir:", {k: int(v) for k, v in rekap.items()})
perlu_perhatian = (terkini[terkini["status"] != "Normal"].groupby("komoditas")["wilayah"].count()
                   .sort_values(ascending=False))
print("Komoditas yang perlu diperhatikan (jumlah wilayah berstatus Waspada/Tinggi):")
print(perlu_perhatian.to_string() if len(perlu_perhatian) else "  - tidak ada -")

In [ ]:
# =============================================================
# 7.2 FUNGSI-FUNGSI KOMPONEN DASHBOARD
# =============================================================
DAFTAR_WILAYAH = sorted(WILAYAH)
BULAN = pd.period_range(df["tanggal"].min(), df["tanggal"].max(), freq="M")
LABEL_BULAN = [f"{NAMA_BULAN[p.month - 1]} {p.year}" for p in BULAN]
MAKS_WILAYAH = 5     # batasan (constraint) agar grafik tidak menjadi "spageti"
GAYA_STATUS = {"Normal":  dict(ikon="✔", latar="#E8F1FA", teks="#0B3D66"),
               "Waspada": dict(ikon="⚠", latar="#FFF4E5", teks="#7A3E00"),
               "Tinggi":  dict(ikon="⛔", latar="#FDECEA", teks="#8A1C1C")}

def status_dari_rasio(r):
    return "Normal" if r <= 1.0 else "Waspada" if r <= 1.10 else "Tinggi"

def saring_data(komoditas, wilayah=None, mulai=None, akhir=None):
    d = df[df["komoditas"] == komoditas]
    if mulai is not None:
        d = d[d["tanggal"] >= pd.Timestamp(mulai)]
    if akhir is not None:
        d = d[d["tanggal"] <= pd.Timestamp(akhir)]
    if wilayah:
        d = d[d["wilayah"].isin(wilayah)]
    return d

def hitung_kpi(d, komoditas):
    acuan = KOMODITAS[komoditas]["acuan"]
    seri = d.groupby("tanggal")["harga"].mean()
    h_akhir = seri.iloc[-1]
    h_lalu = seri.iloc[-5] if len(seri) >= 5 else seri.iloc[0]
    per_wil = d[d["tanggal"] == seri.index[-1]].groupby("wilayah")["harga"].mean().sort_values()
    return dict(tanggal=seri.index[-1], harga=h_akhir, ubah=(h_akhir / h_lalu - 1) * 100,
                termahal=per_wil.index[-1], h_termahal=per_wil.iloc[-1],
                termurah=per_wil.index[0], h_termurah=per_wil.iloc[0],
                status=status_dari_rasio(h_akhir / acuan), acuan=acuan)

def kartu_kpi(k, komoditas):
    sat = KOMODITAS[komoditas]["satuan"]
    g = GAYA_STATUS[k["status"]]
    arah = "▲ naik" if k["ubah"] > 0.05 else "▼ turun" if k["ubah"] < -0.05 else "■ stabil"
    def kartu(judul, isi, sub, latar="#FFFFFF", warna="#1F2937", tepi="#D1D5DB"):
        return (f"<div style='flex:1;min-width:170px;background:{latar};border:1px solid {tepi};"
                f"border-radius:10px;padding:10px 14px;font-family:Arial,sans-serif'>"
                f"<div style='font-size:12px;color:#4B5563'>{judul}</div>"
                f"<div style='font-size:21px;font-weight:700;color:{warna};margin:2px 0'>{isi}</div>"
                f"<div style='font-size:12px;color:#4B5563'>{sub}</div></div>")
    return ("<div style='display:flex;gap:10px;flex-wrap:wrap;margin:6px 0 4px 0'>"
            + kartu("Harga rata-rata provinsi", rupiah(k["harga"]), f"per {sat} · {tgl_id(k['tanggal'])}")
            + kartu("Perubahan 4 minggu", f"{k['ubah']:+.1f}%".replace(".", ","), arah)
            + kartu("Wilayah termahal", k["termahal"], f"{rupiah(k['h_termahal'])} · termurah: {k['termurah']}")
            + kartu("Status terhadap acuan", f"{g['ikon']} {k['status']}",
                    f"acuan {rupiah(k['acuan'])}/{sat}", latar=g["latar"], warna=g["teks"], tepi=g["teks"])
            + "</div>")

def grafik_tren(d, komoditas, wilayah_terpilih):
    acuan, sat = KOMODITAS[komoditas]["acuan"], KOMODITAS[komoditas]["satuan"]
    fig = go.Figure()
    if not wilayah_terpilih:
        agg = d.groupby("tanggal")["harga"].agg(["mean", "min", "max"])
        fig.add_trace(go.Scatter(x=agg.index, y=agg["max"], mode="lines", line=dict(width=0),
                                 name="Tertinggi antarwilayah", hovertemplate="Tertinggi: Rp %{y:,.0f}<extra></extra>"))
        fig.add_trace(go.Scatter(x=agg.index, y=agg["min"], mode="lines", line=dict(width=0), fill="tonexty",
                                 fillcolor="rgba(0,114,178,0.15)", name="Rentang antarwilayah",
                                 hovertemplate="Terendah: Rp %{y:,.0f}<extra></extra>"))
        fig.add_trace(go.Scatter(x=agg.index, y=agg["mean"], mode="lines", line=dict(color=OI["biru"], width=2.5),
                                 name="Rata-rata provinsi", hovertemplate="Rata-rata: Rp %{y:,.0f}<extra></extra>"))
    else:
        for i, w in enumerate(wilayah_terpilih):
            sw = d[d["wilayah"] == w].sort_values("tanggal")
            fig.add_trace(go.Scatter(x=sw["tanggal"], y=sw["harga"], mode="lines", name=w,
                                     line=dict(color=PALET[i % len(PALET)], width=2),
                                     hovertemplate=w + ": Rp %{y:,.0f}<extra></extra>"))
    fig.add_hline(y=acuan, line_dash="dash", line_color="#374151", line_width=1,
                  annotation_text=f"Harga acuan {rupiah(acuan)}", annotation_position="top left")
    fig.update_layout(template="plotly_white", height=430, separators=",.", hovermode="x unified",
                      title=dict(text=f"<b>Tren harga {komoditas}</b> (Rp/{sat}) — arahkan kursor untuk detail", x=0),
                      legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
                      margin=dict(l=10, r=10, t=80, b=10))
    fig.update_xaxes(rangeslider_visible=True,
                     rangeselector=dict(buttons=[dict(count=3, label="3 bln", step="month", stepmode="backward"),
                                                 dict(count=6, label="6 bln", step="month", stepmode="backward"),
                                                 dict(count=1, label="1 thn", step="year", stepmode="backward"),
                                                 dict(step="all", label="Semua")]))
    fig.update_yaxes(tickprefix="Rp ", tickformat=",.0f")
    return fig

def grafik_wilayah(d, komoditas, wilayah_terpilih=None):
    acuan, sat = KOMODITAS[komoditas]["acuan"], KOMODITAS[komoditas]["satuan"]
    t = d["tanggal"].max()
    s_ = d[d["tanggal"] == t].groupby("wilayah")["harga"].mean().sort_values()
    st = [status_dari_rasio(v / acuan) for v in s_.values]
    warna = [WARNA_STATUS[x] if x != "Normal" else "#9CA3AF" for x in st]
    if wilayah_terpilih:   # wilayah terpilih ditegaskan dengan garis tepi tebal
        garis = [3 if w in wilayah_terpilih else 0 for w in s_.index]
    else:
        garis = [0] * len(s_)
    fig = go.Figure(go.Bar(x=s_.values, y=s_.index, orientation="h", marker_color=warna,
                           marker_line=dict(color=TEKS, width=garis),
                           text=[f"{rupiah(v)} · {x}" for v, x in zip(s_.values, st)], textposition="outside",
                           customdata=[(v / acuan - 1) * 100 for v in s_.values],
                           hovertemplate="<b>%{y}</b><br>Rp %{x:,.0f}<br>%{customdata:+.1f}% dari acuan<extra></extra>"))
    fig.add_vline(x=acuan, line_dash="dash", line_color="#374151", line_width=1,
                  annotation_text="acuan", annotation_position="top")
    fig.update_layout(template="plotly_white", height=420, separators=",.",
                      title=dict(text=f"<b>Perbandingan antarwilayah</b>, {tgl_id(t)} (Rp/{sat})", x=0),
                      xaxis=dict(range=[0, s_.max() * 1.35], tickprefix="Rp ", tickformat=",.0f"),
                      margin=dict(l=10, r=10, t=60, b=10))
    return fig

def tabel_detail(d, komoditas, n_minggu=12):
    piv = d.pivot_table(index="tanggal", columns="wilayah", values="harga").sort_index().tail(n_minggu)
    piv.index = [tgl_id(t) for t in piv.index]
    piv.index.name = "Minggu"
    return piv.apply(lambda kol: kol.map(rupiah))

print("Komponen dashboard siap:", ", ".join(["saring_data", "hitung_kpi", "kartu_kpi",
                                               "grafik_tren", "grafik_wilayah", "tabel_detail"]))

### 7.3 Dashboard Interaktif
Jalankan sel berikut, lalu coba ubah komoditas, pilih beberapa wilayah, geser periode, dan centang "Tampilkan tabel detail". Perhatikan baris **umpan balik** di bawah panel filter yang selalu memberi tahu apa yang sedang ditampilkan.

> Tips: pada grafik tren, gunakan tombol **3 bln / 6 bln / 1 thn / Semua** atau geser *range slider* di bawah grafik untuk melakukan *zoom*. Arahkan kursor ke garis atau batang untuk melihat detail angka.

In [ ]:
# =============================================================
# 7.3 DASHBOARD INTERAKTIF SIPANGAN BENGKULU (ipywidgets + plotly)
# =============================================================
gaya = {"description_width": "90px"}
w_kom = widgets.Dropdown(options=[(f"{k} (Rp/{v['satuan']})", k) for k, v in KOMODITAS.items()],
                         value="Cabai Merah Keriting", description="Komoditas", style=gaya,
                         layout=widgets.Layout(width="420px"))
w_wil = widgets.SelectMultiple(options=DAFTAR_WILAYAH, value=(), rows=5, description="Wilayah", style=gaya,
                               layout=widgets.Layout(width="420px"))
w_per = widgets.SelectionRangeSlider(options=list(zip(LABEL_BULAN, BULAN)),
                                     index=(max(0, len(BULAN) - 12), len(BULAN) - 1),
                                     description="Periode", style=gaya, layout=widgets.Layout(width="95%"))
w_det = widgets.Checkbox(value=False, description="Tampilkan tabel detail mingguan (12 minggu terakhir periode)",
                         indent=False)
w_reset = widgets.Button(description="Reset filter", icon="undo", button_style="")
w_umpan = widgets.HTML()
w_out = widgets.Output()
petunjuk = widgets.HTML("<span style='font-size:12px;color:#4B5563'>Wilayah: kosongkan untuk melihat rata-rata "
                        "provinsi; tahan <b>Ctrl</b> (Windows) / <b>Cmd</b> (Mac) untuk memilih lebih dari satu "
                        f"(maksimal {MAKS_WILAYAH}).</span>")
_sedang_reset = {"aktif": False}

def umpan_balik(teks, jenis="ok"):
    warna = {"ok": ("#E8F1FA", "#0B3D66", "✔"), "peringatan": ("#FFF4E5", "#7A3E00", "⚠"),
             "galat": ("#FDECEA", "#8A1C1C", "✘")}[jenis]
    return (f"<div style='background:{warna[0]};color:{warna[1]};padding:6px 10px;border-radius:6px;"
            f"font-family:Arial,sans-serif;font-size:13px'>{warna[2]} {teks}</div>")

def perbarui(_=None):
    if _sedang_reset["aktif"]:
        return
    kom = w_kom.value
    wil = list(w_wil.value)
    p_awal, p_akhir = w_per.value
    pesan_tambahan = ""
    if len(wil) > MAKS_WILAYAH:                                   # constraint + feedback
        pesan_tambahan = (f" Anda memilih {len(wil)} wilayah; hanya {MAKS_WILAYAH} pertama yang digambar "
                          "agar grafik tetap terbaca.")
        wil = wil[:MAKS_WILAYAH]
    d_semua = saring_data(kom, None, p_awal.start_time, p_akhir.end_time)
    d = saring_data(kom, wil or None, p_awal.start_time, p_akhir.end_time)
    with w_out:
        clear_output(wait=True)
        if d.empty:
            w_umpan.value = umpan_balik("Tidak ada data untuk kombinasi filter ini. Coba perluas periode.", "galat")
            return
        display(HTML(kartu_kpi(hitung_kpi(d_semua, kom), kom)))
        grafik_tren(d, kom, wil).show()
        grafik_wilayah(d_semua, kom, wil).show()
        if w_det.value:
            display(HTML(f"<b>Detail harga mingguan {kom} (Rp/{KOMODITAS[kom]['satuan']})</b>"))
            display(tabel_detail(d, kom))
    cakupan = ", ".join(wil) if wil else "rata-rata 10 kabupaten/kota"
    jumlah = f"{len(d):,}".replace(",", ".")
    teks = (f"Menampilkan {jumlah} data harga <b>{kom}</b> · {cakupan} · "
            f"{LABEL_BULAN[list(BULAN).index(p_awal)]} – {LABEL_BULAN[list(BULAN).index(p_akhir)]}.")
    w_umpan.value = umpan_balik(teks + pesan_tambahan, "peringatan" if pesan_tambahan else "ok")

def reset(_):
    _sedang_reset["aktif"] = True
    w_kom.value = "Cabai Merah Keriting"; w_wil.value = ()
    w_per.index = (max(0, len(BULAN) - 12), len(BULAN) - 1); w_det.value = False
    _sedang_reset["aktif"] = False
    perbarui()

for w in (w_kom, w_wil, w_per, w_det):
    w.observe(perbarui, names="value")
w_reset.on_click(reset)

judul = widgets.HTML("<div style='font-family:Arial,sans-serif;background:#0B3D66;color:white;padding:10px 14px;"
                     "border-radius:8px'><span style='font-size:20px;font-weight:700'>SIPANGAN Bengkulu</span>"
                     "<span style='font-size:13px'> &nbsp;·&nbsp; Dashboard Pemantauan Harga Pangan "
                     "(data simulasi)</span></div>")
panel_filter = widgets.VBox([widgets.HBox([w_kom, w_reset]), w_wil, petunjuk, w_per, w_det],
                            layout=widgets.Layout(border="1px solid #D1D5DB", padding="8px", margin="6px 0"))
display(widgets.VBox([judul, panel_filter, w_umpan, w_out]))
perbarui()

In [ ]:
# =============================================================
# 7.4 CUPLIKAN STATIS DASHBOARD (untuk laporan / lampiran PNG)
# =============================================================
def cuplikan_dashboard(komoditas, n_bulan=12, simpan=None):
    p_awal, p_akhir = BULAN[-n_bulan], BULAN[-1]
    d = saring_data(komoditas, None, p_awal.start_time, p_akhir.end_time)
    k = hitung_kpi(d, komoditas)
    sat, acuan = KOMODITAS[komoditas]["satuan"], KOMODITAS[komoditas]["acuan"]

    fig = plt.figure(figsize=(15, 8.6))
    gs = fig.add_gridspec(3, 4, height_ratios=[0.5, 2.2, 0.08], hspace=0.35, wspace=0.25, top=0.9)
    fig.suptitle(f"SIPANGAN Bengkulu — {komoditas} (Rp/{sat}), {LABEL_BULAN[-n_bulan]} – {LABEL_BULAN[-1]}",
                 x=0.06, ha="left", fontsize=14, fontweight="bold", color="#0B3D66")
    arah = "naik" if k["ubah"] > 0.05 else "turun" if k["ubah"] < -0.05 else "stabil"
    g = GAYA_STATUS[k["status"]]
    isi_kpi = [("Harga rata-rata provinsi", rupiah(k["harga"]), tgl_id(k["tanggal"]), "white", TEKS),
               ("Perubahan 4 minggu", f"{k['ubah']:+.1f}%".replace(".", ","), arah, "white", TEKS),
               ("Wilayah termahal", k["termahal"], rupiah(k["h_termahal"]), "white", TEKS),
               ("Status terhadap acuan", k["status"].upper(), f"acuan {rupiah(acuan)}", g["latar"], g["teks"])]
    for i, (jdl, nilai, sub, latar, warna) in enumerate(isi_kpi):
        a = fig.add_subplot(gs[0, i]); a.axis("off")
        a.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0.02", transform=a.transAxes,
                                   facecolor=latar, edgecolor=warna if latar != "white" else "#9CA3AF", clip_on=False))
        a.text(0.05, 0.78, jdl, fontsize=9.5, color="#4B5563", transform=a.transAxes)
        a.text(0.05, 0.38, nilai, fontsize=16, fontweight="bold", color=warna, transform=a.transAxes)
        a.text(0.05, 0.10, sub, fontsize=9, color="#4B5563", transform=a.transAxes)

    a1 = fig.add_subplot(gs[1, :3])
    agg = d.groupby("tanggal")["harga"].agg(["mean", "min", "max"])
    a1.fill_between(agg.index, agg["min"], agg["max"], color=OI["biru"], alpha=0.15, label="Rentang antarwilayah")
    a1.plot(agg.index, agg["mean"], color=OI["biru"], lw=2.2, label="Rata-rata provinsi")
    a1.axhline(acuan, color=TEKS, ls="--", lw=1)
    a1.text(agg.index[0], acuan, f" harga acuan {rupiah(acuan)}", va="bottom", fontsize=8.5)
    a1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: rupiah(x)))
    a1.legend(frameon=False, loc="upper left", fontsize=9); format_bulan(a1, 2)
    a1.set_title("Tren harga mingguan")

    a2 = fig.add_subplot(gs[1, 3])
    s_ = d[d["tanggal"] == d["tanggal"].max()].groupby("wilayah")["harga"].mean().sort_values()
    st = [status_dari_rasio(v / acuan) for v in s_.values]
    a2.barh(s_.index, s_.values, color=[WARNA_STATUS[x] if x != "Normal" else "#9CA3AF" for x in st], height=0.65)
    a2.axvline(acuan, color=TEKS, ls="--", lw=1)
    for i, v in enumerate(s_.values):
        a2.text(v * 0.97, i, f"{v/1000:.1f} rb".replace(".", ","), va="center", ha="right", fontsize=8,
                color="white", fontweight="bold")
    a2.text(acuan, len(s_) - 0.4, " acuan", fontsize=8, va="bottom")
    a2.set_xlim(0, s_.max() * 1.3); a2.set_xticks([])
    a2.spines["bottom"].set_visible(False); a2.tick_params(axis="y", length=0, labelsize=8.5)
    a2.set_title("Per wilayah (minggu terakhir)")

    a3 = fig.add_subplot(gs[2, :]); a3.axis("off")
    a3.text(0, 0, "Warna batang: abu-abu = Normal · oranye = Waspada (≤10% di atas acuan) · merah bata = Tinggi "
            "(>10% di atas acuan). Data simulasi untuk keperluan studi kasus.", fontsize=8.5, color="#4B5563")
    if simpan:
        plt.savefig(simpan, bbox_inches="tight")
    plt.show()

cuplikan_dashboard("Cabai Merah Keriting", simpan="output/dashboard_cabai.png")
cuplikan_dashboard("Beras Medium", simpan="output/dashboard_beras.png")

## 8. Tahap 7 — Evaluasi *Usability*

### 8.1 Desain Pengujian
Evaluasi dilakukan dengan desain **within-subject**: setiap responden mencoba dua versi antarmuka secara bergantian.

| Versi | Deskripsi |
|---|---|
| **v1 — Desain lama** | Spreadsheet mingguan, pencarian dengan mengetik nama komoditas, grafik dekoratif seperti grafik (A) |
| **v2 — SIPANGAN** | Dashboard hasil perancangan pada Tahap 6 |

- **Responden:** 10 orang (3 analis dinas, 4 pedagang/masyarakat umum, 3 mahasiswa) sesuai persona.
- **Tugas:** T1–T5 dari Tahap 1.
- **Metrik** (mengacu ISO 9241-11):
  - *Efektivitas* — tingkat keberhasilan tugas (%) dan jumlah kesalahan.
  - *Efisiensi* — waktu penyelesaian tugas (detik).
  - *Kepuasan* — *System Usability Scale* (SUS, 10 butir, skala Likert 1–5).
- **Analisis:** uji-t berpasangan (α = 0,05) dan ukuran efek Cohen's *d*.

> Catatan: data hasil uji pada notebook ini adalah **data simulasi** yang dibangkitkan untuk mendemonstrasikan cara pengolahan metrik *usability*. Pada pengujian sungguhan, sel 8.2 cukup diganti dengan data hasil observasi.

In [ ]:
# =============================================================
# 8.2 DATA HASIL UJI USABILITY (SIMULASI)
# =============================================================
rng_uji = np.random.default_rng(42)
responden = pd.DataFrame({
    "id": [f"R{i:02d}" for i in range(1, 11)],
    "kelompok": ["Analis dinas"] * 3 + ["Pedagang/masyarakat"] * 4 + ["Mahasiswa"] * 3,
})
faktor_kelompok = {"Analis dinas": 1.0, "Pedagang/masyarakat": 1.35, "Mahasiswa": 0.85}

# parameter populasi: peluang berhasil, rata-rata waktu (detik), rata-rata jumlah kesalahan
PARAM = {
    "v1": {"T1": (0.80, 45, 1.0), "T2": (0.60, 95, 1.5), "T3": (0.50, 120, 2.0), "T4": (0.40, 150, 2.5), "T5": (0.70, 80, 1.2)},
    "v2": {"T1": (0.97, 14, 0.1), "T2": (0.95, 26, 0.2), "T3": (0.93, 30, 0.3), "T4": (0.92, 33, 0.3), "T5": (0.78, 55, 0.9)},
}
baris_uji = []
for _, r in responden.iterrows():
    f = faktor_kelompok[r["kelompok"]]
    for versi, tugas_v in PARAM.items():
        for kode, (p, mu, lam) in tugas_v.items():
            p_eff = p if r["kelompok"] != "Pedagang/masyarakat" else p * 0.92
            baris_uji.append({"id": r["id"], "kelompok": r["kelompok"], "versi": versi, "tugas": kode,
                              "berhasil": int(rng_uji.random() < p_eff),
                              "waktu_detik": round(float(rng_uji.lognormal(np.log(mu * f), 0.25)), 1),
                              "kesalahan": int(rng_uji.poisson(lam * f))})
uji = pd.DataFrame(baris_uji)

# ---- Kuesioner SUS: butir ganjil positif, butir genap negatif ----
BUTIR_SUS = ["Saya ingin sering menggunakan sistem ini", "Sistem ini terlalu rumit",
             "Sistem ini mudah digunakan", "Saya butuh bantuan teknisi untuk memakainya",
             "Fungsi-fungsinya terintegrasi dengan baik", "Terlalu banyak inkonsistensi",
             "Kebanyakan orang akan cepat belajar memakainya", "Sistem ini merepotkan",
             "Saya percaya diri memakainya", "Saya perlu banyak belajar sebelum memakainya"]
baris_sus = []
for _, r in responden.iterrows():
    for versi, (mu_s, sd_s) in {"v1": (0.42, 0.10), "v2": (0.80, 0.07)}.items():
        laten = np.clip(rng_uji.normal(mu_s, sd_s), 0, 1)
        jawaban = {}
        for i in range(10):
            nilai = 1 + 4 * laten if i % 2 == 0 else 5 - 4 * laten
            jawaban[f"Q{i+1}"] = int(np.clip(np.round(nilai + rng_uji.normal(0, 0.6)), 1, 5))
        baris_sus.append({"id": r["id"], "kelompok": r["kelompok"], "versi": versi, **jawaban})
sus = pd.DataFrame(baris_sus)

print(f"Data uji tugas : {len(uji)} baris (10 responden × 2 versi × 5 tugas)")
print(f"Data kuesioner : {len(sus)} baris (10 responden × 2 versi)")
display(uji.head(6))
display(sus.head(4))

In [ ]:
# =============================================================
# 8.3 METRIK EFEKTIVITAS & EFISIENSI PER TUGAS
# =============================================================
metrik = (uji.groupby(["tugas", "versi"])
             .agg(keberhasilan_pct=("berhasil", lambda x: 100 * x.mean()),
                  waktu_median_detik=("waktu_detik", "median"),
                  rata_kesalahan=("kesalahan", "mean"))
             .round(1).unstack("versi"))
metrik.columns = [f"{m} ({v})" for m, v in metrik.columns]
metrik["Penghematan waktu (%)"] = (100 * (1 - metrik["waktu_median_detik (v2)"] / metrik["waktu_median_detik (v1)"])).round(1)
metrik.insert(0, "Tugas", tugas.set_index("Kode").loc[metrik.index, "Tugas"].values)
display(metrik)

ringkas = uji.groupby("versi").agg(keberhasilan=("berhasil", "mean"), waktu=("waktu_detik", "median"),
                                   kesalahan=("kesalahan", "mean"))
print(f"Keberhasilan tugas keseluruhan : v1 {ringkas.loc['v1','keberhasilan']*100:.0f}%  →  v2 {ringkas.loc['v2','keberhasilan']*100:.0f}%")
print(f"Median waktu per tugas         : v1 {ringkas.loc['v1','waktu']:.0f} dtk  →  v2 {ringkas.loc['v2','waktu']:.0f} dtk")
print(f"Rata-rata kesalahan per tugas  : v1 {ringkas.loc['v1','kesalahan']:.2f}  →  v2 {ringkas.loc['v2','kesalahan']:.2f}")

In [ ]:
# =============================================================
# 8.4 PERHITUNGAN SKOR SUS
# Skor butir ganjil = jawaban - 1 ; butir genap = 5 - jawaban ; total × 2,5
# =============================================================
def skor_sus(baris):
    total = sum((baris[f"Q{i}"] - 1) if i % 2 == 1 else (5 - baris[f"Q{i}"]) for i in range(1, 11))
    return total * 2.5

def kategori_sus(s):           # rentang penerimaan (Bangor, Kortum & Miller, 2008)
    return "Acceptable" if s >= 70 else "Marginal" if s >= 50 else "Not acceptable"

def adjektiva_sus(s):          # skala adjektiva (Bangor, Kortum & Miller, 2009)
    batas = [(85.5, "Excellent"), (72.6, "Good"), (52.0, "OK"), (39.2, "Poor"), (0, "Worst imaginable")]
    return next(lbl for b, lbl in batas if s >= b)

sus["skor_SUS"] = sus.apply(skor_sus, axis=1)
tabel_sus = sus.pivot(index=["id", "kelompok"], columns="versi", values="skor_SUS").reset_index()
tabel_sus["selisih"] = tabel_sus["v2"] - tabel_sus["v1"]
display(tabel_sus)

ringkasan_sus = pd.DataFrame({
    "Versi": ["v1 — Desain lama", "v2 — SIPANGAN"],
    "Rata-rata SUS": [tabel_sus["v1"].mean(), tabel_sus["v2"].mean()],
    "Simpangan baku": [tabel_sus["v1"].std(), tabel_sus["v2"].std()],
})
ringkasan_sus["Kategori penerimaan"] = ringkasan_sus["Rata-rata SUS"].map(kategori_sus)
ringkasan_sus["Adjektiva"] = ringkasan_sus["Rata-rata SUS"].map(adjektiva_sus)
display(ringkasan_sus.round(1))

# ---- Uji statistik berpasangan ----
t_sus, p_sus = stats.ttest_rel(tabel_sus["v2"], tabel_sus["v1"])
d_sus = tabel_sus["selisih"].mean() / tabel_sus["selisih"].std()
waktu_resp = uji.groupby(["id", "versi"])["waktu_detik"].sum().unstack()
t_wkt, p_wkt = stats.ttest_rel(waktu_resp["v1"], waktu_resp["v2"])
d_wkt = (waktu_resp["v1"] - waktu_resp["v2"]).mean() / (waktu_resp["v1"] - waktu_resp["v2"]).std()

def tafsir(p):
    return "SIGNIFIKAN" if p < 0.05 else "tidak signifikan"

print(f"SUS  : t(9) = {t_sus:.2f}, p = {p_sus:.5f} → {tafsir(p_sus)} pada α = 0,05; Cohen's d = {d_sus:.2f}")
print(f"Waktu total 5 tugas: v1 {waktu_resp['v1'].mean():.0f} dtk vs v2 {waktu_resp['v2'].mean():.0f} dtk; "
      f"t(9) = {t_wkt:.2f}, p = {p_wkt:.5f} → {tafsir(p_wkt)}; Cohen's d = {d_wkt:.2f}")

In [ ]:
# =============================================================
# 8.5 VISUALISASI HASIL EVALUASI
# =============================================================
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
kode_t = list(tugas["Kode"])
x = np.arange(len(kode_t)); lebar = 0.38
w1, w2 = ABU, OI["biru"]

# (a) Keberhasilan per tugas
kb = uji.groupby(["tugas", "versi"])["berhasil"].mean().unstack() * 100
ax[0].bar(x - lebar/2, kb["v1"], lebar, color=w1, label="v1 — Desain lama")
ax[0].bar(x + lebar/2, kb["v2"], lebar, color=w2, label="v2 — SIPANGAN")
for i in range(len(x)):
    ax[0].text(x[i] - lebar/2, kb["v1"].iloc[i] + 1.5, f"{kb['v1'].iloc[i]:.0f}", ha="center", fontsize=8)
    ax[0].text(x[i] + lebar/2, kb["v2"].iloc[i] + 1.5, f"{kb['v2'].iloc[i]:.0f}", ha="center", fontsize=8, color=w2)
ax[0].set_xticks(x, kode_t); ax[0].set_ylim(0, 125); ax[0].set_ylabel("Keberhasilan (%)")
ax[0].legend(frameon=False, fontsize=8.5, loc="upper left", ncol=2)
ax[0].set_title("(a) Tingkat keberhasilan tugas")

# (b) Waktu penyelesaian (dot plot per responden + median)
for i, k in enumerate(kode_t):
    for versi, geser, warna in (("v1", -0.18, w1), ("v2", 0.18, w2)):
        nilai = uji[(uji["tugas"] == k) & (uji["versi"] == versi)]["waktu_detik"]
        ax[1].scatter(np.full(len(nilai), i + geser) + rng_uji.uniform(-0.05, 0.05, len(nilai)), nilai,
                      s=14, color=warna, alpha=0.6)
        ax[1].plot([i + geser - 0.12, i + geser + 0.12], [nilai.median()] * 2, color=TEKS, lw=2)
ax[1].set_xticks(range(len(kode_t)), kode_t); ax[1].set_ylabel("Waktu (detik)")
ax[1].set_title("(b) Waktu penyelesaian (garis = median)")
ax[1].text(0.98, 0.95, "abu-abu = v1\nbiru = v2", transform=ax[1].transAxes, ha="right", va="top", fontsize=8.5)

# (c) Skor SUS per responden (slope chart: before-after)
for _, r in tabel_sus.iterrows():
    ax[2].plot([0, 1], [r["v1"], r["v2"]], color=OI["biru"] if r["selisih"] > 0 else OI["merah_bata"],
               alpha=0.6, marker="o", ms=4)
for y_, lbl in ((50, "Marginal"), (70, "Acceptable")):
    ax[2].axhline(y_, color=ABU, ls=":", lw=1); ax[2].text(1.04, y_, lbl, fontsize=8, va="center", color="#4B5563")
ax[2].plot([0, 1], [tabel_sus["v1"].mean(), tabel_sus["v2"].mean()], color=TEKS, lw=3, marker="s", ms=7,
           label="Rata-rata")
ax[2].text(-0.06, tabel_sus["v1"].mean(), f"{tabel_sus['v1'].mean():.1f}", ha="right", va="center", fontweight="bold")
ax[2].text(1.06, tabel_sus["v2"].mean() + 4, f"{tabel_sus['v2'].mean():.1f}", ha="left", va="center", fontweight="bold")
ax[2].set_xticks([0, 1], ["v1 — Desain lama", "v2 — SIPANGAN"]); ax[2].set_xlim(-0.3, 1.35); ax[2].set_ylim(0, 100)
ax[2].legend(frameon=False, fontsize=8.5, loc="lower right")
ax[2].set_title("(c) Skor SUS tiap responden")
plt.tight_layout(); plt.savefig("output/hasil_evaluasi.png", bbox_inches="tight"); plt.show()

In [ ]:
# =============================================================
# 8.6 DAFTAR PERIKSA PRINSIP USABILITY PADA SIPANGAN (v2)
# =============================================================
periksa = pd.DataFrame([
    ("Visibilitas", "Panel filter, KPI, dan grafik tampil sekaligus; tidak ada fitur utama yang tersembunyi", "7.3"),
    ("Umpan balik", "Baris status ✔/⚠/✘ diperbarui setiap kali filter berubah; nilai muncul saat hover", "7.3"),
    ("Konsistensi", "Format Rupiah, satuan, warna status, dan istilah sama di kartu, grafik, dan heatmap", "7.1–7.4"),
    ("Batasan / pencegahan kesalahan", "Dropdown (tidak bisa salah ketik), slider periode tidak bisa terbalik, maks. 5 wilayah", "3.1, 7.3"),
    ("Kemudahan penggunaan", "Tombol Reset filter, pilihan cepat 3 bln/6 bln/1 thn, label dalam bahasa sehari-hari", "7.3"),
    ("Recognition over recall", "Semua pilihan ditampilkan sebagai menu; pengguna tidak perlu menghafal nama baku", "3.1"),
    ("Gestalt", "Pengelompokan kontrol dalam panel berbingkai; warna status yang sama untuk makna yang sama", "3.2, 7.3"),
    ("Aksesibilitas", "Palet Okabe-Ito, kontras label status ≥ 4,5:1, status ditandai ikon dan teks", "6.3"),
    ("Tufte (data-ink)", "Tanpa 3D, gradasi, atau arsiran; sumbu dari nol; label langsung", "6.1, 7.4"),
    ("Shneiderman", "Overview (heatmap & KPI) → zoom & filter (widget, range slider) → detail (hover & tabel)", "7"),
], columns=["Prinsip", "Penerapan pada SIPANGAN", "Bagian notebook"])
periksa.insert(2, "Status", "✔ Terpenuhi")
display(periksa)

## 9. Tahap 8 — Iterasi Desain Berdasarkan Hasil Evaluasi

Desain berpusat pada pengguna bersifat **berulang**: hasil evaluasi tidak hanya membuktikan bahwa desain baru lebih baik, tetapi juga menunjukkan bagian yang masih perlu diperbaiki. Sel berikut mencari tugas dengan kinerja terendah pada v2 secara otomatis, lalu menetapkan rencana perbaikan untuk iterasi berikutnya.

In [ ]:
# =============================================================
# 9. IDENTIFIKASI TUGAS TERLEMAH PADA v2 & RENCANA PERBAIKAN
# =============================================================
v2 = uji[uji["versi"] == "v2"].groupby("tugas").agg(keberhasilan=("berhasil", "mean"),
                                                    waktu=("waktu_detik", "median"),
                                                    kesalahan=("kesalahan", "mean"))
# skor masalah gabungan: makin rendah keberhasilan, makin lama, makin banyak salah -> makin bermasalah
v2["skor_masalah"] = ((1 - v2["keberhasilan"]) / (1 - v2["keberhasilan"]).max()
                      + v2["waktu"] / v2["waktu"].max() + v2["kesalahan"] / v2["kesalahan"].max()).round(2)
v2 = v2.sort_values("skor_masalah", ascending=False)
display(v2.assign(keberhasilan=lambda t: (t["keberhasilan"] * 100).round(0)).round(2))

RENCANA = {
    "T1": "Perbesar kartu harga rata-rata dan tampilkan juga harga di wilayah pengguna secara default.",
    "T2": "Jadikan tombol '6 bln' sebagai tampilan awal grafik tren dan beri anotasi otomatis pada titik puncak.",
    "T3": "Tambahkan opsi pengurutan (termahal/termurah) dan sorot wilayah terpilih pada grafik batang.",
    "T4": "Tambahkan daftar peringatan 'Komoditas berstatus Tinggi' di atas heatmap dengan tautan ke dashboard.",
    "T5": "Ganti kotak centang 'Tampilkan tabel detail' dengan tombol 'Lihat detail ▸' yang terlihat jelas di bawah "
          "grafik, dan izinkan klik pada batang wilayah untuk langsung membuka detail wilayah tersebut.",
}
terlemah = v2.index[0]
print(f"Tugas dengan kinerja terendah pada v2 : {terlemah} — {tugas.set_index('Kode').loc[terlemah, 'Tugas']}")
print(f"  keberhasilan {v2.loc[terlemah, 'keberhasilan']*100:.0f}%, median waktu {v2.loc[terlemah, 'waktu']:.0f} dtk, "
      f"rata-rata kesalahan {v2.loc[terlemah, 'kesalahan']:.1f}")
print(f"Rencana perbaikan iterasi berikutnya (v3): {RENCANA[terlemah]}")
print()
print("Prioritas perbaikan seluruh tugas (dari skor masalah tertinggi):")
for i, kode in enumerate(v2.index, 1):
    print(f"  {i}. {kode}: {RENCANA[kode]}")

In [ ]:
# =============================================================
# 10. RINGKASAN HASIL UNTUK KESIMPULAN
# =============================================================
hasil_akhir = pd.DataFrame({
    "Metrik": ["Keberhasilan tugas (%)", "Median waktu per tugas (detik)", "Rata-rata kesalahan per tugas",
               "Skor SUS rata-rata", "Kategori SUS"],
    "v1 — Desain lama": [round(ringkas.loc["v1", "keberhasilan"] * 100), round(ringkas.loc["v1", "waktu"]),
                         round(ringkas.loc["v1", "kesalahan"], 2), round(tabel_sus["v1"].mean(), 1),
                         f"{kategori_sus(tabel_sus['v1'].mean())} ({adjektiva_sus(tabel_sus['v1'].mean())})"],
    "v2 — SIPANGAN": [round(ringkas.loc["v2", "keberhasilan"] * 100), round(ringkas.loc["v2", "waktu"]),
                      round(ringkas.loc["v2", "kesalahan"], 2), round(tabel_sus["v2"].mean(), 1),
                      f"{kategori_sus(tabel_sus['v2'].mean())} ({adjektiva_sus(tabel_sus['v2'].mean())})"],
})
display(hasil_akhir)
print("Berkas gambar yang dihasilkan (folder output/):")
for f in sorted(os.listdir("output")):
    print("  -", f)

## 10. Kesimpulan

1. **Analisis kebutuhan berbasis pengguna** (persona, daftar tugas, serta kebutuhan fungsional dan non-fungsional) membuat setiap fitur dashboard dapat ditelusuri ke kebutuhan nyata pengguna. Fitur tidak ditambahkan hanya karena keinginan pengembang.
2. **Psikologi kognitif** terbukti berpengaruh langsung. Simulasi *Recognition over Recall* menunjukkan bahwa sebagian besar ketikan pengguna gagal dikenali sistem pencarian berbasis teks persis, sedangkan menu pilihan menghilangkan kesalahan tersebut. Prinsip Gestalt (kedekatan, kesamaan, wilayah bersama) dipakai untuk menyusun panel filter, kartu KPI, dan pewarnaan status yang konsisten.
3. **Prinsip visualisasi Tufte** mencegah distorsi. Grafik dengan sumbu terpotong memiliki *Lie Factor* jauh di atas 1, sedangkan grafik yang diperbaiki menyajikan proporsi sesuai data. Grafik garis tepat untuk tren waktu dan grafik batang terurut tepat untuk perbandingan wilayah.
4. **Mantra Shneiderman** diwujudkan dalam dashboard SIPANGAN: *overview* melalui heatmap dan kartu KPI, *zoom & filter* melalui widget dan *range slider*, serta *details-on-demand* melalui *hover* dan tabel detail.
5. **Aksesibilitas** diuji secara kuantitatif: rasio kontras label status memenuhi WCAG AA, dan warna status berbasis palet Okabe-Ito tetap dapat dibedakan pada simulasi buta warna, berbeda dengan warna "lampu lalu lintas" yang hijau dan merahnya hampir tidak dapat dibedakan.
6. **Evaluasi usability** (tabel ringkasan pada bagian 10) menunjukkan desain baru meningkatkan keberhasilan tugas, mempercepat waktu penyelesaian, mengurangi kesalahan, dan menaikkan skor SUS dari kategori *Not acceptable* ke *Acceptable* dengan perbedaan yang signifikan secara statistik.
7. Evaluasi juga mengungkap bagian yang masih lemah (akses ke detail mingguan, T5), sehingga perbaikan dijadwalkan pada iterasi berikutnya. Hal ini menegaskan bahwa perancangan antarmuka adalah **proses berulang**: analisis kebutuhan → konsep → prototipe → evaluasi → perbaikan.

Secara keseluruhan, studi kasus ini menunjukkan bahwa kualitas sistem informasi tidak hanya ditentukan oleh fungsi teknis, tetapi juga oleh kemampuannya menyajikan informasi secara **jelas, mudah dipahami, dan mudah diakses** oleh beragam pengguna.

---

## 11. Daftar Pustaka

1. Bangor, A., Kortum, P. T., & Miller, J. T. (2008). An empirical evaluation of the System Usability Scale. *International Journal of Human–Computer Interaction, 24*(6), 574–594.
2. Bangor, A., Kortum, P., & Miller, J. (2009). Determining what individual SUS scores mean: Adding an adjective rating scale. *Journal of Usability Studies, 4*(3), 114–123.
3. Brooke, J. (1996). SUS: A "quick and dirty" usability scale. Dalam P. W. Jordan dkk. (Ed.), *Usability Evaluation in Industry* (hlm. 189–194). Taylor & Francis.
4. ISO 9241-11:2018. *Ergonomics of human-system interaction — Part 11: Usability: Definitions and concepts*. International Organization for Standardization.
5. Machado, G. M., Oliveira, M. M., & Fernandes, L. A. F. (2009). A physiologically-based model for simulation of color vision deficiency. *IEEE Transactions on Visualization and Computer Graphics, 15*(6), 1291–1298.
6. Nielsen, J. (1993). *Usability Engineering*. Morgan Kaufmann.
7. Norman, D. A. (2013). *The Design of Everyday Things* (Revised and expanded edition). Basic Books.
8. Okabe, M., & Ito, K. (2008). *Color Universal Design (CUD): How to make figures and presentations that are friendly to colorblind people*. J*Fly Data Depository.
9. Shneiderman, B. (1996). The eyes have it: A task by data type taxonomy for information visualizations. *Proceedings of the IEEE Symposium on Visual Languages*, 336–343.
10. Tufte, E. R. (2001). *The Visual Display of Quantitative Information* (2nd ed.). Graphics Press.
11. W3C. (2018). *Web Content Accessibility Guidelines (WCAG) 2.1*. World Wide Web Consortium.
12. Wertheimer, M. (1923). Untersuchungen zur Lehre von der Gestalt II. *Psychologische Forschung, 4*, 301–350.
13. Febriandirza, A. (2026). *Human-Computer Interaction (HCI) dan Data Visualization* [Materi kuliah praktisi]. Program Studi S1 Teknik Informatika, Universitas Bengkulu.